# Lost at the Switch: Colab setup

**Before running:** Runtime → Change runtime type → **A100** (or **L4** if A100 is unavailable; a T4 is too small for the 9B model).

Run the cells top to bottom. Each one prints what it did. Paste the output of the last cell back to Claude.

Tokens: if you've added `GITHUB_TOKEN` and `HF_TOKEN` under the key icon (with notebook access on), they're used automatically. If not, the token cell will ask you to paste them.

## 1. Check the GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch
assert torch.cuda.is_available(), "No GPU: go to Runtime -> Change runtime type and pick A100 or L4"
gpu = torch.cuda.get_device_name(0); mem = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"GPU: {gpu} ({mem:.0f} GB)")
if mem < 22: print("WARNING: under 22 GB, the Qwen 9B + 0.8B pair will not fit. Switch to A100 or L4.")

## 2. Mount Google Drive (results are saved here so they survive disconnects)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE_DIR = '/content/drive/MyDrive/lost-at-the-switch'
import os; os.makedirs(f'{DRIVE_DIR}/results', exist_ok=True)
print('Results will be saved to', DRIVE_DIR + '/results')

## 3. Tokens

In [ ]:
import os
from getpass import getpass
def get_secret(name, prompt):
    try:
        from google.colab import userdata
        v = userdata.get(name)
        if v: print(f'{name}: loaded from Colab secrets'); return v
    except Exception:
        pass
    return getpass(prompt)
GITHUB_TOKEN = get_secret('GITHUB_TOKEN', 'Paste GitHub token (read access to Narpear/lost-at-the-switch): ')
os.environ['HF_TOKEN'] = get_secret('HF_TOKEN', 'Paste Hugging Face token: ')
print('Tokens set.')

## 4. Clone (or update) the repo and install

In [ ]:
REPO = '/content/lost-at-the-switch'
if os.path.isdir(f'{REPO}/.git'):
    !git -C {REPO} pull --ff-only
else:
    !git clone https://{GITHUB_TOKEN}@github.com/Narpear/lost-at-the-switch.git {REPO}
%cd {REPO}
!git log --oneline -1
!pip install -q -e ".[dev]"
# Results go to Drive; model weights go to fast local disk
!rm -rf results && ln -s {DRIVE_DIR}/results results
os.environ['HF_HOME'] = '/content/hf_cache'
print('Installed. results/ ->', os.path.realpath('results'))

## 5. Unit tests (about 1 minute, all 31 should pass)

In [ ]:
!python -m pytest -q 2>&1 | tail -5

## 6. Sanity check on the real Qwen pair

The first run downloads about 20 GB of weights (5-10 minutes). It checks: shared tokenizer, KV-cache crop vs recompute, T=0 speculative == greedy, and measures the cost ratio c. **Paste the output of this cell to Claude.**

In [ ]:
!python scripts/sanity_check.py --pair qwen 2>&1 | grep -v -E "Loading weights|Fetching|it/s" | tee results/sanity_qwen.txt